# Lab 22-03 · Boost, Buck-Boost and SEPIC

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import topologies as tp


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The boost

*Predict* the duty cycle, inductor ripple and output ripple for 3.6 V to 5 V at 1.5 A, 1 MHz, 1 µH and 22 µF.

In [2]:
D = tp.boost_duty(3.6, 5.0)
print(f"D = {D:.3f}, I_L = {tp.boost_inductor_current(1.5, D):.3f} A, ripple {tp.boost_ripple_current(3.6, D, 1e6, 1e-6):.3f} A, "
      f"output ripple {tp.boost_output_ripple(1.5, D, 1e6, 22e-6)*1e3:.2f} mV, C_out RMS {tp.boost_cout_rms(1.5, D):.3f} A")
print(f"RHP zero at {tp.boost_rhp_zero_hz(D, 5/1.5, 1e-6)/1e3:.0f} kHz")
s = ngspice("boost.cir")
print({k: s[k] for k in ("voa", "dia", "dva", "vob", "voc")})

D = 0.280, I_L = 2.083 A, ripple 1.008 A, output ripple 19.09 mV, C_out RMS 0.935 A
RHP zero at 275 kHz


{'voa': 4.999011, 'dia': 1.007775, 'dva': 0.019077, 'vob': 4.544508, 'voc': 7.140405}


## Part 2 · The gain limit

*Predict* the gain at D = 0.8 and 0.9 with R_L = 0.1 Ω into 25 Ω, and where the gain peaks.

In [3]:
m, dstar = tp.boost_max_gain(0.1, 25)
print(f"max gain {m:.2f} at D = {dstar:.3f}")
for d in (0.5, 0.8, 0.9):
    print(f"D = {d}: M = {tp.boost_ratio_with_rl(d, 0.1, 25):.3f}, efficiency {tp.boost_efficiency_with_rl(d, 0.1, 25)*100:.1f} %")
d = np.linspace(0, 0.99, 500)
plt.figure()
plt.plot(d, 1 / (1 - d), ls=":", label="ideal 1/(1 - D)")
plt.plot(d, [tp.boost_ratio_with_rl(x, 0.1, 25) for x in d], label="R_L/R = 0.004")
plt.ylim(0, 12); plt.xlabel("D"); plt.ylabel("M"); plt.legend(); plt.show()

max gain 7.91 at D = 0.937
D = 0.5: M = 1.969, efficiency 98.4 %
D = 0.8: M = 4.545, efficiency 90.9 %
D = 0.9: M = 7.143, efficiency 71.4 %


## Part 3 · The SEPIC

*Predict* the output and the coupling capacitor's average from 12 V at D = 0.5 into 6 Ω, with and without 50 mΩ windings.

In [4]:
for r in (1e-9, 0.05):
    vo, vc, i1, i2 = tp.sepic_dc(12.0, 0.5, 6.0, r, r)
    print(f"windings {r*1e3:.0f} mohm: V_out {vo:.3f} V, V_C1 {vc:.3f} V, I_L1 {i1:.3f} A, I_L2 {i2:.3f} A")
s = ngspice("sepic.cir")
print({k: s[k] for k in ("vout", "vc1", "il1")})

windings 0 mohm: V_out 12.000 V, V_C1 12.000 V, I_L1 2.000 A, I_L2 2.000 A
windings 50 mohm: V_out 11.803 V, V_C1 12.000 V, I_L1 1.967 A, I_L2 1.967 A


{'vout': 11.80121, 'vc1': 11.99953, 'il1': 1.966498}


## Part 4 · Light load

*Predict* the open-loop output of the 3.6 V boost at 50 mA (K = 0.02) with D = 0.28.

In [5]:
print(f"K_crit(0.28) = {tp.boost_k_crit(0.28):.4f}; DCM M = {tp.boost_dcm_ratio(0.28, 0.02):.3f}, V = {3.6*tp.boost_dcm_ratio(0.28, 0.02):.2f} V")
k = np.logspace(-3, 0, 400)
plt.figure()
for d in (0.2, 0.33, 0.6):
    plt.semilogx(k, [tp.boost_ratio_any(d, x) for x in k], label=f"D = {d}")
plt.ylim(1, 8); plt.xlabel("K = 2L/(R T_s)"); plt.ylabel("M"); plt.legend(); plt.show()
dd = np.linspace(0, 1, 300)
plt.figure(); plt.plot(dd, tp.boost_k_crit(dd)); plt.xlabel("D"); plt.ylabel("K_crit = D D'^2"); plt.show()

K_crit(0.28) = 0.1452; DCM M = 2.542, V = 9.15 V


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.